In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import poisson


results = pd.read_csv('archive (1)/results.csv')
elo = pd.read_csv('eloratings.csv')
pastnames = pd.read_csv('archive (1)/former_names.csv')
shootouts = pd.read_csv('archive (1)/shootouts.csv')


result_teams = set(results["home_team"].unique()) | set(results["away_team"].unique())

uniqueTeams = set(results[results["date"] >= "2000-01-01"]["home_team"].unique()) | set(results[results["date"] >= "2000-01-01"]["away_team"].unique())
print(len(uniqueTeams))

shootouts["date"] = pd.to_datetime(shootouts["date"])
shootouts = shootouts.set_index(["date", "home_team"])

TeamNames = pd.unique(pd.concat([results["home_team"], results["away_team"]])).tolist()
Tournaments = pd.unique(results["tournament"]).tolist()




In [ ]:
elo["team"] = elo["team"].str.replace('\xa0', ' ', regex=False)

name_mapping = {
    "China": "China PR",
    "Czechia": "Czech Republic",
    "Ireland": "Republic of Ireland",
    "Macao": "Macau",
    "Macedonia": "North Macedonia",
    "East Germany": "German DR",
    "Democratic Republic of Congo": "DR Congo",
    "Congo-Brazzaville": "Congo",
    "East Timor": "Timor-Leste",
    "Reunion": "Réunion",
    "Sao Tome and Principe": "São Tomé and Príncipe",
    "US Virgin Islands": "United States Virgin Islands",
    "Wallis and Futuna": "Wallis Islands and Futuna",
    "Vatican": "Vatican City",
    "Soviet Union": "Russia",
    "West Germany": "Germany",
    "Saint Barthelemy": "Saint Barthélemy",
    "Khmer Republic": "Cambodia",
    "North Yemen": "Yemen",
    "South Vietnam": "Vietnam", 
    "Surinam": "Suriname",
    "Eastern Samoa": "American Samoa",
    "Federated States of Micronesia": "Micronesia",
}



elo["team"] = elo["team"].replace(name_mapping)

former_mapping = dict(zip(pastnames["former"], pastnames["current"]))
elo["team"] = elo["team"].replace(former_mapping)

elo_teams = set(elo["team"].unique())
result_teams = set(results["home_team"].unique()) | set(results["away_team"].unique())


only_in_elo = elo_teams - result_teams
only_in_results = result_teams - elo_teams



print("In Elo but not results:", sorted(only_in_elo))
print("In results but not Elo:", sorted(only_in_results))

elo["team"] = elo["team"].map({team: i for i, team in enumerate(TeamNames)}).fillna(-1).astype(int)

In [ ]:
# Set up results
results = results.dropna(subset=["home_score", "away_score"]).reset_index(drop=True)

# Calculate days since last match for each team (vectorized)
results["date"] = pd.to_datetime(results["date"])
results_home = results[["date", "home_team"]].rename(columns={"home_team": "team"})
results_away = results[["date", "away_team"]].rename(columns={"away_team": "team"})
all_matches = pd.concat([results_home, results_away]).sort_values(["team", "date"]).reset_index(drop=True)
all_matches["prev_date"] = all_matches.groupby("team")["date"].shift(1)
all_matches["days_since_last"] = (all_matches["date"] - all_matches["prev_date"]).dt.days

home_days = all_matches.rename(columns={"team": "home_team", "days_since_last": "DaysSinceLastTeam1"})[["date", "home_team", "DaysSinceLastTeam1"]]
away_days = all_matches.rename(columns={"team": "away_team", "days_since_last": "DaysSinceLastTeam2"})[["date", "away_team", "DaysSinceLastTeam2"]]

results = results.merge(home_days, on=["date", "home_team"], how="left")
results = results.merge(away_days, on=["date", "away_team"], how="left")

results["DaysSinceLastTeam1"] = results["DaysSinceLastTeam1"].fillna(-1)
results["DaysSinceLastTeam2"] = results["DaysSinceLastTeam2"].fillna(-1)

dates = results["date"][5:]

cleaned = {"Date": dates.reset_index(drop=True),
           "Team1": [],
           "Team2": [],
           "Tournament": [],
           "Neutral": [],
           "DaysSinceLastTeam1": results["DaysSinceLastTeam1"][5:].reset_index(drop=True),
           "DaysSinceLastTeam2": results["DaysSinceLastTeam2"][5:].reset_index(drop=True),
           "h2hTeamOne": [],
           "h2hTeamTwo": [],
           "RFteamOne": [],
           "RFteamTwo": [],
           "RGFteamOne": [],
           "RGAteamOne": [],
           "RGFteamTwo": [],
           "RGAteamTwo": [],
           "shootoutWinner": [],
           "Winner": []}



for j in range(5, len(dates)+5):

    date = dates[j]

    resultsBeforeDate = results[:j]
    teamOne = results["home_team"][j]
    teamTwo = results["away_team"][j]

    teamOneGames = resultsBeforeDate[
        (resultsBeforeDate['home_team'] == teamOne) |
        (resultsBeforeDate['away_team'] == teamOne)].copy()
    teamOneGames['location'] = np.where(
        teamOneGames['home_team'] == teamOne, 'home', 'away')

    teamTwoGames = resultsBeforeDate[
        (resultsBeforeDate['home_team'] == teamTwo) |
        (resultsBeforeDate['away_team'] == teamTwo)].copy()
    teamTwoGames['location'] = np.where(
        teamTwoGames['home_team'] == teamTwo, 'home', 'away')

    h2hresults = resultsBeforeDate[resultsBeforeDate["date"] >= (pd.to_datetime(dates[j]) - pd.DateOffset(years=2)).strftime("%Y-%m-%d")].copy()
    h2hCount = len(h2hresults[
        ((h2hresults['home_team'] == teamOne) & (h2hresults['away_team'] == teamTwo)) |
        ((h2hresults['away_team'] == teamOne) & (h2hresults['home_team'] == teamTwo))].copy())

    winner = 0
    if results["home_score"][j] > results["away_score"][j]:
        winner = 1
    elif results["home_score"][j] < results["away_score"][j]:
        winner = 2

    teamOneDets = [0,0,0,0]  # gf, ga, rf, h2h
    teamTwoDets = [0,0,0,0]
    teamOneGamesPlayedWTeamTwo = 0
    teamTwoGamesPlayedWTeamOne = 0
    teamOneRecentGamesCount = 0
    teamTwoRecentGamesCount = 0

    for i in range(len(teamOneGames)-1, -1, -1):
        teamOneGame = teamOneGames.index[i]
        home_score = resultsBeforeDate["home_score"][teamOneGame]
        away_score = resultsBeforeDate["away_score"][teamOneGame]
        is_home = teamOneGames["location"].iloc[i] == "home"
        team_score = home_score if is_home else away_score
        opp_score = away_score if is_home else home_score
        opponent = (teamOneGames["away_team"].iloc[i] if is_home else teamOneGames["home_team"].iloc[i])

        if (len(teamOneGames)-5 <= i) if len(teamOneGames)-5 > 0 else True:
            if team_score > opp_score:
                teamOneDets[2] += 3
            elif team_score == opp_score:
                teamOneDets[2] += 1
            teamOneDets[0] += team_score
            teamOneDets[1] += opp_score
            teamOneRecentGamesCount += 1

        if opponent == teamTwo and teamOneGamesPlayedWTeamTwo < 5 and teamOneGamesPlayedWTeamTwo < h2hCount:
            if team_score > opp_score:
                teamOneDets[3] += 3
            elif team_score == opp_score:
                teamOneDets[3] += 1
            teamOneGamesPlayedWTeamTwo += 1

        if (teamOneGamesPlayedWTeamTwo >= h2hCount or teamOneGamesPlayedWTeamTwo >= 5) and teamOneRecentGamesCount > 4:
            break

    for i in range(len(teamTwoGames)-1, -1, -1):
        teamTwoGame = teamTwoGames.index[i]
        home_score = resultsBeforeDate["home_score"][teamTwoGame]
        away_score = resultsBeforeDate["away_score"][teamTwoGame]
        is_home = teamTwoGames["location"].iloc[i] == "home"
        team_score = home_score if is_home else away_score
        opp_score = away_score if is_home else home_score
        opponent = (teamTwoGames["away_team"].iloc[i] if is_home else teamTwoGames["home_team"].iloc[i])

        if (len(teamTwoGames)-5 <= i) if len(teamTwoGames)-5 > 0 else True:
            if team_score > opp_score:
                teamTwoDets[2] += 3
            elif team_score == opp_score:
                teamTwoDets[2] += 1
            teamTwoRecentGamesCount += 1
            teamTwoDets[0] += team_score
            teamTwoDets[1] += opp_score

        if opponent == teamOne and teamTwoGamesPlayedWTeamOne < 5 and teamTwoGamesPlayedWTeamOne < h2hCount:
            if team_score > opp_score:
                teamTwoDets[3] += 3
            elif team_score == opp_score:
                teamTwoDets[3] += 1
            teamTwoGamesPlayedWTeamOne += 1

        if (teamTwoGamesPlayedWTeamOne >= h2hCount or teamTwoGamesPlayedWTeamOne >= 5) and teamTwoRecentGamesCount > 4:
            break

    cleaned["Team1"].append(TeamNames.index(teamOne))
    cleaned["Team2"].append(TeamNames.index(teamTwo))
    cleaned["Tournament"].append(Tournaments.index(results["tournament"][j]))
    cleaned["Neutral"].append(int(results["neutral"][j]))

    cleaned["h2hTeamOne"].append(teamOneDets[3]/teamOneGamesPlayedWTeamTwo if teamOneGamesPlayedWTeamTwo > 0 else -1)
    cleaned["h2hTeamTwo"].append(teamTwoDets[3]/teamTwoGamesPlayedWTeamOne if teamTwoGamesPlayedWTeamOne > 0 else -1)
    cleaned["RFteamOne"].append(teamOneDets[2]/teamOneRecentGamesCount if teamOneRecentGamesCount > 0 else -1)
    cleaned["RFteamTwo"].append(teamTwoDets[2]/teamTwoRecentGamesCount if teamTwoRecentGamesCount > 0 else -1)
    cleaned["RGFteamOne"].append(teamOneDets[0]/teamOneRecentGamesCount if teamOneRecentGamesCount > 0 else 0)
    cleaned["RGAteamOne"].append(teamOneDets[1]/teamOneRecentGamesCount if teamOneRecentGamesCount > 0 else 0)
    cleaned["RGFteamTwo"].append(teamTwoDets[0]/teamTwoRecentGamesCount if teamTwoRecentGamesCount > 0 else 0)
    cleaned["RGAteamTwo"].append(teamTwoDets[1]/teamTwoRecentGamesCount if teamTwoRecentGamesCount > 0 else 0)
    cleaned["Winner"].append(winner)

    if winner == 0:
        shootout = shootouts["winner"].get((date, teamOne), "NONE")
        if shootout != "NONE":
            if shootout == teamOne:
                cleaned["shootoutWinner"].append(1)
            else:
                cleaned["shootoutWinner"].append(2)
        else:
            cleaned["shootoutWinner"].append(np.nan)
    else:
        cleaned["shootoutWinner"].append(winner)
    

    if j % 1000 == 0:
        print(j)

cleanedDF = pd.DataFrame(cleaned)

In [ ]:
#set up elo
cleanedDF["Date"] = pd.to_datetime(cleanedDF["Date"])
elo["date"] = pd.to_datetime(elo["date"], format="mixed")

cleanedDF = cleanedDF.sort_values("Date").reset_index(drop=True)
elo = elo.sort_values("date").reset_index(drop=True)

# Merge Elo for Team1
elo_team1 = elo.rename(columns={"team": "Team1", "rating": "EloTeam1"})[["date", "Team1", "EloTeam1"]]
cleanedDF = pd.merge_asof(cleanedDF, elo_team1, left_on="Date", right_on="date", by="Team1")
cleanedDF = cleanedDF.drop(columns=["date"])

# Merge Elo for Team2
elo_team2 = elo.rename(columns={"team": "Team2", "rating": "EloTeam2"})[["date", "Team2", "EloTeam2"]]
cleanedDF = pd.merge_asof(cleanedDF, elo_team2, left_on="Date", right_on="date", by="Team2")
cleanedDF = cleanedDF.drop(columns=["date"])

# Drop rows where Elo is missing (the 155 unrated teams)
cleanedDF = cleanedDF.dropna(subset=["EloTeam1", "EloTeam2"])

print(cleanedDF.shape)
print(cleanedDF[["Date", "Team1", "EloTeam1", "Team2", "EloTeam2"]].head())



In [ ]:
np.random.seed(67)
mask = np.random.rand(len(cleanedDF)) < 0.5


cleanedDF.loc[mask, ["EloTeam1", "EloTeam2"]] = cleanedDF.loc[mask, ["EloTeam2", "EloTeam1"]].values

cleanedDF.loc[mask, ["Team1", "Team2"]] = cleanedDF.loc[mask, ["Team2", "Team1"]].values
cleanedDF.loc[mask, ["h2hTeamOne", "h2hTeamTwo"]] = cleanedDF.loc[mask, ["h2hTeamTwo", "h2hTeamOne"]].values

cleanedDF.loc[mask, ["RFteamOne", "RFteamTwo"]] = cleanedDF.loc[mask, ["RFteamTwo", "RFteamOne"]].values

cleanedDF.loc[mask, ["RGFteamOne", "RGFteamTwo"]] = cleanedDF.loc[mask, ["RGFteamTwo", "RGFteamOne"]].values
cleanedDF.loc[mask, ["RGAteamOne", "RGAteamTwo"]] = cleanedDF.loc[mask, ["RGAteamTwo", "RGAteamOne"]].values

cleanedDF.loc[mask, "Winner"] = np.where(cleanedDF.loc[mask, "Winner"] == 1, 2, 
                                         np.where(cleanedDF.loc[mask, "Winner"] == 2, 1, 0)
)




In [7]:
from scipy.stats import poisson
import numpy as np

global_avg = 1.4722596879718168

def poisson_probabilities(row):
    lambda1 = row["RGFteamOne"] * row["RGAteamTwo"] / global_avg
    lambda2 = row["RGFteamTwo"] * row["RGAteamOne"] / global_avg
    
    # handle missing data
    if lambda1 <= 0 or lambda2 <= 0:
        return pd.Series({"PoissonTeam1Win": -1, "PoissonDraw": -1, "PoissonTeam2Win": -1})
    
    max_goals = 7
    team1_win = 0
    draw = 0
    team2_win = 0
    
    for i in range(max_goals):
        for j in range(max_goals):
            p =  poisson.pmf(i, lambda1) * poisson.pmf(j, lambda2)
            if i > j:
                team1_win += p
            elif i == j:
                draw += p
            else:
                team2_win += p
    
    return pd.Series({"PoissonTeam1Win": team1_win, "PoissonDraw": draw, "PoissonTeam2Win": team2_win})

cleanedDF[["PoissonTeam1Win", "PoissonDraw", "PoissonTeam2Win"]] = cleanedDF.apply(poisson_probabilities, axis=1)

print(cleanedDF[["PoissonTeam1Win", "PoissonDraw", "PoissonTeam2Win"]].head())

   PoissonTeam1Win  PoissonDraw  PoissonTeam2Win
0        -1.000000    -1.000000        -1.000000
1         0.125333     0.137211         0.693670
2        -1.000000    -1.000000        -1.000000
3         0.015570     0.038403         0.768733
4        -1.000000    -1.000000        -1.000000


In [8]:
import numpy as np

lineups = pd.read_csv('players/game_lineups.csv', low_memory=False)
valuations = pd.read_csv('players/player_valuations.csv', low_memory=False)
players = pd.read_csv('players/players.csv', low_memory=False)
games = pd.read_csv('players/games.csv', low_memory=False)

# Join nationality onto valuations
val_with_country = valuations[["player_id", "date", "market_value_in_eur"]].merge(
    players[["player_id", "name", "country_of_citizenship"]], on="player_id", how="left"
)

val_with_country["date"] = pd.to_datetime(val_with_country["date"])
val_with_country["year"] = val_with_country["date"].dt.year

# Take each player's most recent valuation per year
latest = val_with_country.sort_values("date").groupby(["player_id", "year"]).tail(1)

# For each player, create a "shadow" row for the next year
# so their valuation is also visible when computing year+1's squad
latest_next = latest.copy()
latest_next["year"] = latest_next["year"] + 1

# Combine real + shadow rows, then deduplicate:
# for any (player, year), keep only the most recent actual date
combined = pd.concat([latest, latest_next], ignore_index=True)
combined = combined.sort_values("date").groupby(["player_id", "year"]).tail(1)

# Top 23 average per country per year
squad_yearly = (
    combined.groupby(["country_of_citizenship", "year"])
    .apply(lambda x: x.nlargest(23, "market_value_in_eur")["market_value_in_eur"].mean())
    .reset_index()
    .rename(columns={0: "squad_value"})
)

# Log transform before normalising (squad values are exponentially distributed)
squad_yearly["log_value"] = np.log1p(squad_yearly["squad_value"])
squad_yearly["squad_value_normalized"] = squad_yearly.groupby("year")["log_value"].transform(
    lambda x: (x - x.mean()) / x.std() if len(x) >= 10 else np.nan
)
squad_yearly["squad_value_normalized"] = squad_yearly["squad_value_normalized"].fillna(-1).clip(-3, 3)

print(squad_yearly["squad_value_normalized"].describe())

# Fix name mismatches
sq_name_mapping = {
    "Bosnia-Herzegovina": "Bosnia and Herzegovina",
    "Brunei Darussalam": "Brunei",
    "China": "China PR",
    "Cote d'Ivoire": "Ivory Coast",
    "Curacao": "Curaçao",
    "Ireland": "Republic of Ireland",
    "Korea, North": "North Korea",
    "Korea, South": "South Korea",
    "Macao": "Macau",
    "Neukaledonien": "New Caledonia",
    "Saint-Martin": "Saint Martin",
    "Sao Tome and Principe": "São Tomé and Príncipe",
    "Southern Sudan": "South Sudan",
    "St. Kitts & Nevis": "Saint Kitts and Nevis",
    "St. Lucia": "Saint Lucia",
    "The Gambia": "Gambia",
    "Türkiye": "Turkey"
}
squad_yearly["country_of_citizenship"] = squad_yearly["country_of_citizenship"].replace(sq_name_mapping)

# Fake date column (Jan 1 of each year) for merge_asof
squad_yearly["date"] = pd.to_datetime(squad_yearly["year"].astype(str) + "-01-01")
squad_yearly = squad_yearly.sort_values("date").reset_index(drop=True)

# Add team name columns
cleanedDF = cleanedDF.sort_values("Date").reset_index(drop=True)
cleanedDF["Team1_name"] = cleanedDF["Team1"].apply(lambda x: TeamNames[x] if x >= 0 else None)
cleanedDF["Team2_name"] = cleanedDF["Team2"].apply(lambda x: TeamNames[x] if x >= 0 else None)

sq_teams = set(squad_yearly["country_of_citizenship"].unique())
cleaned_teams = set(cleanedDF["Team1_name"].dropna().unique())
print("Matching teams:", len(cleaned_teams.intersection(sq_teams)))

# Merge Team1
sv_team1 = squad_yearly[["date", "country_of_citizenship", "squad_value_normalized"]].rename(
    columns={"country_of_citizenship": "Team1_name", "squad_value_normalized": "SquadValueTeam1"}
)
cleanedDF = pd.merge_asof(cleanedDF, sv_team1, left_on="Date", right_on="date", by="Team1_name")
cleanedDF = cleanedDF.drop(columns=["date"])

# Merge Team2
sv_team2 = squad_yearly[["date", "country_of_citizenship", "squad_value_normalized"]].rename(
    columns={"country_of_citizenship": "Team2_name", "squad_value_normalized": "SquadValueTeam2"}
)
cleanedDF = pd.merge_asof(cleanedDF, sv_team2, left_on="Date", right_on="date", by="Team2_name")
cleanedDF = cleanedDF.drop(columns=["date"])

cleanedDF["SquadValueTeam1"] = cleanedDF["SquadValueTeam1"].fillna(-1)
cleanedDF["SquadValueTeam2"] = cleanedDF["SquadValueTeam2"].fillna(-1)

cleanedDF = cleanedDF.drop(columns=["Team1_name", "Team2_name"])

# Verify Spain > Ecuador
for team in ["Spain", "Ecuador", "Argentina"]:
    idx = TeamNames.index(team)
    rows = cleanedDF[(cleanedDF["Team1"] == idx) | (cleanedDF["Team2"] == idx)].tail(1)
    if len(rows):
        r = rows.iloc[0]
        sv = r["SquadValueTeam1"] if r["Team1"] == idx else r["SquadValueTeam2"]
        print(f"{team}: squad z = {sv:.2f}")

count    3638.000000
mean       -0.001223
std         0.996335
min        -3.000000
25%        -0.671722
50%        -0.032907
75%         0.727274
max         2.409860
Name: squad_value_normalized, dtype: float64
Matching teams: 180
Spain: squad z = 1.77
Ecuador: squad z = 1.13
Argentina: squad z = 1.72


In [9]:
tournament_importance = {
    # Tier 5 - Major finals
    'FIFA World Cup': 5,
    'UEFA Euro': 5,
    'Copa América': 5,
    'African Cup of Nations': 5,
    'AFC Asian Cup': 5,
    'CONCACAF Championship': 5,
    'Gold Cup': 5,
    'Oceania Nations Cup': 5,
    'Olympic Games': 5,

    # Tier 4 - Secondary majors / confederations cups
    'Confederations Cup': 4,
    'UEFA Nations League': 4,
    'CONCACAF Nations League': 4,
    'CONMEBOL–UEFA Cup of Champions': 4,
    'Copa América qualification': 4,
    'AFC Challenge Cup': 4,
    'EAFF Championship': 4,
    'AFF Championship': 4,
    'COSAFA Cup': 4,
    'CECAFA Cup': 4,
    'Gulf Cup': 4,
    'WAFF Championship': 4,
    'SAFF Cup': 4,
    'CFU Caribbean Cup': 4,
    'CAFA Nations Cup': 4,
    'Superclásico de las Américas': 4,
    'Dynasty Cup': 4,
    'British Home Championship': 4,
    'Arab Cup': 4,
    'Asian Games': 4,
    'Pan American Championship': 4,
    'NAFC Championship': 4,
    'NAFU Championship': 4,
    'Melanesia Cup': 4,
    'Pacific Games': 4,
    'All-African Games': 4,
    'Bolivarian Games': 4,

    # Tier 3 - Qualifiers for major tournaments
    'FIFA World Cup qualification': 3,
    'UEFA Euro qualification': 3,
    'African Cup of Nations qualification': 3,
    'AFC Asian Cup qualification': 3,
    'CONCACAF Championship qualification': 3,
    'Gold Cup qualification': 3,
    'Oceania Nations Cup qualification': 3,
    'CONCACAF Nations League qualification': 3,
    'AFC Challenge Cup qualification': 3,
    'EAFF Championship qualification': 3,
    'AFF Championship qualification': 3,
    'ASEAN Championship qualification': 3,
    'COSAFA Cup qualification': 3,
    'CFU Caribbean Cup qualification': 3,
    'Arab Cup qualification': 3,
    'Copa América qualification': 3,
    'CONIFA World Cup qualification': 3,
    'CONIFA World Football Cup qualification': 3,

    # Tier 2 - Minor regional tournaments
    'ABCS Tournament': 2,
    'AFC Solidarity Cup': 2,
    'AFF Championship qualification': 2,
    'ASEAN Championship': 2,
    'African Friendship Games': 2,
    'Afro-Asian Games': 2,
    'Al Ain International Cup': 2,
    'Amílcar Cabral Cup': 2,
    'Atlantic Cup': 2,
    'Atlantic Heritage Cup': 2,
    'Balkan Cup': 2,
    'Baltic Cup': 2,
    'CCCF Championship': 2,
    'CONCACAF Series': 2,
    'CONIFA Africa Football Cup': 2,
    'CONIFA Asia Cup': 2,
    'CONIFA European Football Cup': 2,
    'CONIFA South America Football Cup': 2,
    'CONIFA World Football Cup': 2,
    'Central American and Caribbean Games': 2,
    'Central European International Cup': 2,
    'Copa Rio Branco': 2,
    'Copa Roca': 2,
    'Copa Oswaldo Cruz': 2,
    'Copa del Pacífico': 2,
    'ELF Cup': 2,
    'East Asian Games': 2,
    'Far Eastern Championship Games': 2,
    'FIFI Wild Cup': 2,
    'GaNEFo': 2,
    'Indian Ocean Island Games': 2,
    'Inter-Allied Games': 2,
    'Intercontinental Cup': 2,
    'Island Games': 2,
    'Kirin Cup': 2,
    'Kirin Challenge Cup': 2,
    'Lunar New Year Cup': 2,
    'Melanesia Cup': 2,
    'Merdeka Tournament': 2,
    'Mundialito': 2,
    'Muratti Vase': 2,
    'Nordic Championship': 2,
    'Oceania Nations Cup qualification': 2,
    'Pacific Mini Games': 2,
    'Rous Cup': 2,
    'SAFF Cup': 2,
    'South Asian Games': 2,
    'South Pacific Games': 2,
    'South Pacific Mini Games': 2,
    'Southeast Asian Games': 2,
    'Southeast Asian Peninsular Games': 2,
    'TIFOCO Tournament': 2,
    'UDEAC Cup': 2,
    'UNCAF Cup': 2,
    'UNIFFAC Cup': 2,
    'Viva World Cup': 2,
    'West African Cup': 2,
    'ConIFA Challenger Cup': 2,
    'Copa Artigas': 2,
    "Copa Bernardo O'Higgins": 2,
    'Copa Carlos Dittborn': 2,
    'Copa Chevallier Boutell': 2,
    'Copa Confraternidad': 2,
    'Copa Félix Bogado': 2,
    'Copa Juan Pinto Durán': 2,
    'Copa Lipton': 2,
    'Copa Newton': 2,
    'Copa Paz del Chaco': 2,
    'Copa Premio Honor Argentino': 2,
    'Copa Premio Honor Uruguayo': 2,
    'Copa Ramón Castilla': 2,
    'International Tournament of Peoples, Cultures and Tribes': 2,
    'The Other Final': 2,
    'Nile Basin Tournament': 2,
    'MSG Prime Minister\'s Cup': 2,

    # Tier 1 - Friendlies and invitational tournaments
    'Friendly': 1,
    'Beijing International Friendship Tournament': 1,
    'Benedikt Fontana Cup': 1,
    'Brazil Independence Cup': 1,
    'Canadian Shield': 1,
    'Corsica Cup': 1,
    "Coupe de l'Outre-Mer": 1,
    'Cup of Ancient Civilizations': 1,
    'Cyprus International Tournament': 1,
    'Dakar Tournament': 1,
    'Diamond Jubilee International Football Tournament': 1,
    'Dragon Cup': 1,
    'Dunhill Cup': 1,
    'EAFF Championship qualification': 1,
    'FIFA 75th Anniversary Cup': 1,
    'FIFA Series': 1,
    'Four Nations Tournament': 1,
    "Four Nations' Cup": 1,
    'Gold Cup qualification': 1,
    'Great Wall Cup': 1,
    'Guangzhou International Friendship Tournament': 1,
    'Hungary Heritage Cup': 1,
    'Indonesia Tournament': 1,
    'Inter Games': 1,
    'Joe Robbie Cup': 1,
    'Jordan International Tournament': 1,
    'King Hassan II Tournament': 1,
    "King's Cup": 1,
    'Korea Cup': 1,
    'Kuneitra Cup': 1,
    'Mahinda Rajapaksa Cup': 1,
    'Malta International Tournament': 1,
    'Mapinduzi Cup': 1,
    'Marianas Cup': 1,
    'Marlboro Cup': 1,
    'Matthews Cup': 1,
    'Mauritius Four Nations Cup': 1,
    'Merlion Cup': 1,
    'Miami Cup': 1,
    'Millennium Cup': 1,
    'Morocco, Capital of African Football': 1,
    'Mukuru 4 Nations': 1,
    'Navruz Cup': 1,
    'Nehru Cup': 1,
    'Niamh Challenge Cup': 1,
    'Nations Cup': 1,
    'OSN Cup': 1,
    'Open International Championship': 1,
    'Outrigger Challenge Cup': 1,
    'Pacific Mini Games': 1,
    'Palestine Cup': 1,
    'Palestine International Championship': 1,
    'Peace Cup': 1,
    'Philippine Peace Cup': 1,
    'Phillip Seaga Cup': 1,
    "Prime Minister's Cup": 1,
    'Real Madrid 75th Anniversary Cup': 1,
    'SKN Football Festival': 1,
    'Scania 100 Tournament': 1,
    'Simba Tournament': 1,
    'Soccer Ashes': 1,
    'South Asian Super Cup': 1,
    'Three Nations Cup': 1,
    'Tournament Burkina Faso': 1,
    'Tournoi de France': 1,
    'Trans-Tasman Cup': 1,
    'Tri Nation Tournament': 1,
    'Tri-Nations Series': 1,
    'Tynwald Hill Tournament': 1,
    'UNIFFAC Cup': 1,
    'USA Cup': 1,
    'United Arab Emirates Friendship Tournament': 1,
    'Unity Cup': 1,
    'VFF Cup': 1,
    'Vietnam Independence Cup': 1,
    'World Unity Cup': 1,
    'Zambian Independence Tournament': 1,
    'Évence Coppée Trophy': 1,
    'Windward Islands Tournament': 1,
    'NAFU Championship': 1,
    'Afro-Asian Games': 1,
    'Nile Basin Tournament': 1,
    'Inter-Allied Games': 1,
}
results["TournamentImportance"] = results["tournament"].map(tournament_importance).fillna(1).astype(int)
cleanedDF["TournamentImportance"] = results["tournament"].map(tournament_importance).fillna(1).astype(int)[5:].reset_index(drop=True)
cleanedDF = cleanedDF.drop(columns=["Tournament"])

In [10]:
cleanedDF["EloDiff"] = cleanedDF["EloTeam1"] - cleanedDF["EloTeam2"]
cleanedDF["SquadValueDiff"] = cleanedDF["SquadValueTeam1"] - cleanedDF["SquadValueTeam2"]

In [11]:


cleanedDF.to_csv('output.csv', index=False)    

# cleanedDF = pd.read_csv('output.csv',low_memory=False)    



In [ ]:
trainingData = cleanedDF[(cleanedDF["Date"] >= "1900-01-01") & (cleanedDF["Date"] <= "2024-01-01")]
testingData = cleanedDF[cleanedDF["Date"] > "2024-01-01"]

X_train = trainingData.drop(columns=["Date", "Winner", "h2hTeamOne", "h2hTeamTwo", "TournamentImportance","PoissonTeam1Win", "PoissonDraw", "PoissonTeam2Win"])
X_test = testingData.drop(columns=["Date", "Winner", "h2hTeamOne", "h2hTeamTwo", "TournamentImportance","PoissonTeam1Win", "PoissonDraw", "PoissonTeam2Win"])

Y_train = trainingData["Winner"]
Y_test = testingData["Winner"]

filtered = cleanedDF[(cleanedDF["Date"] >= "2004-01-01")]

X = filtered.drop(columns=["Date", "Winner", "Neutral"])
Y = filtered["Winner"]


In [13]:
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.utils.class_weight import compute_sample_weight

split = TimeSeriesSplit()
xgb = XGBClassifier(eval_metric='mlogloss')

recentDF = cleanedDF[(cleanedDF["Date"] >= "2013-01-01") & (cleanedDF["Date"] <= "2024-01-01")]

params = {
    "n_estimators": [200, 225, 250],
    "max_depth": [3, 5, 10],
    "learning_rate": [0.05, 0.01, 0.1]
}

X = recentDF.drop(columns=["Date", "Winner", "shootoutWinner","PoissonTeam1Win", "PoissonDraw", "PoissonTeam2Win"])
Y = recentDF["Winner"]

GroupStageAI = GridSearchCV(xgb, params, cv=split, n_jobs=2)
GroupStageAI.fit(X, Y, sample_weight=compute_sample_weight('balanced', Y))

print(GroupStageAI.best_params_)
print(GroupStageAI.best_score_)
print(pd.Series(GroupStageAI.best_estimator_.feature_importances_, index=X.columns).sort_values(ascending=False))

{'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 250}
0.527265130640752
EloDiff                 0.428102
EloTeam1                0.064663
SquadValueDiff          0.064424
RGAteamTwo              0.048881
RFteamTwo               0.041739
RGAteamOne              0.040007
Team2                   0.031307
Neutral                 0.031054
EloTeam2                0.029541
Team1                   0.027706
RGFteamOne              0.026945
SquadValueTeam2         0.026460
SquadValueTeam1         0.026262
DaysSinceLastTeam2      0.021474
h2hTeamOne              0.019478
DaysSinceLastTeam1      0.019241
TournamentImportance    0.019088
RGFteamTwo              0.018985
RFteamOne               0.014644
h2hTeamTwo              0.000000
dtype: float32


In [14]:
from sklearn.metrics import classification_report


testingData = cleanedDF[cleanedDF["Date"] > "2024-01-01"]

X_test = testingData.drop(columns=["Date", "Winner", "shootoutWinner", 
                            "PoissonTeam1Win", "PoissonDraw", "PoissonTeam2Win"])

Y_test = testingData["Winner"]

proba = GroupStageAI.best_estimator_.predict_proba(X_test)

# Ignore draw probability, normalise between team1 and team2
knockout_preds = []
for p in proba:
    team1_prob = p[1] / (p[1] + p[2])
    knockout_preds.append(0 if team1_prob > 0.5 else 1)

# Remap Y_test to binary (drop draws)
binary_mask = Y_test != 0
binary_Y = Y_test[binary_mask].map({1: 0, 2: 1})
binary_preds = [knockout_preds[i] for i in range(len(Y_test)) if Y_test.iloc[i] != 0]

print(classification_report(binary_Y, binary_preds))

              precision    recall  f1-score   support

           0       0.79      0.81      0.80       911
           1       0.81      0.79      0.80       941

    accuracy                           0.80      1852
   macro avg       0.80      0.80      0.80      1852
weighted avg       0.80      0.80      0.80      1852



In [15]:
from sklearn.metrics import classification_report

testingData = cleanedDF[cleanedDF["Date"] > "2024-01-01"]

X_test = testingData.drop(columns=["Date", "Winner", "shootoutWinner", 
                            "PoissonTeam1Win", "PoissonDraw", "PoissonTeam2Win"])

Y_test = testingData["Winner"]

proba = GroupStageAI.best_estimator_.predict_proba(X_test)

actualDrawRate= np.mean(Y_test == 0)

predictedDrawRate = 1
draw_threshold = 0.374

# while np.abs(predictedDrawRate-actualDrawRate) > 0.01:
#     predictions = []
#     for p in proba:
#         if p[0] > draw_threshold:
#             predictions.append(0)
#         else:
#             predictions.append(1 if p[1] > p[2] else 2)

#     predictions = np.array(predictions)
#     predictedDrawRate = np.mean(predictions == 0)

#     if predictedDrawRate-actualDrawRate > 0:
#         draw_threshold += 0.001
#     else:
#         draw_threshold -= 0.001

#     print(draw_threshold)
predictions = []
for p in proba:
    if p[0] > draw_threshold:
        predictions.append(0)
    else:
        predictions.append(1 if p[1] > p[2] else 2)

predictions = np.array(predictions)
predictedDrawRate = np.mean(predictions == 0)

if predictedDrawRate-actualDrawRate > 0:
    draw_threshold += 0.001
else:
    draw_threshold -= 0.001


print(classification_report(Y_test, predictions))


print("Draw prediction rate:", np.mean(predictions == 0))
print("Actual draw rate:", actualDrawRate)

print(draw_threshold)




              precision    recall  f1-score   support

           0       0.30      0.30      0.30       581
           1       0.66      0.66      0.66       911
           2       0.66      0.65      0.65       941

    accuracy                           0.57      2433
   macro avg       0.54      0.54      0.54      2433
weighted avg       0.57      0.57      0.57      2433

Draw prediction rate: 0.24003288121660502
Actual draw rate: 0.238799835593917
0.375


In [ ]:
# import joblib
# import numpy as np
# import pandas as pd

# # Save model first
# joblib.dump(GroupStageAI.best_estimator_, 'group_stage_model.pkl')

# def get_team_features(team_name, opponent_name, is_neutral=True):
#     team_idx = TeamNames.index(team_name)
#     opp_idx = TeamNames.index(opponent_name)
    
#     # Get most recent match for team as Team1
#     as_team1 = cleanedDF[cleanedDF["Team1"] == team_idx].tail(1)
#     # Get most recent match for team as Team2
#     as_team2 = cleanedDF[cleanedDF["Team2"] == team_idx].tail(1)
    
#     # Pick whichever is more recent
#     if len(as_team1) == 0 and len(as_team2) == 0:
#         return None
#     elif len(as_team1) == 0:
#         last = as_team2.iloc[0]
#         is_home = False
#     elif len(as_team2) == 0:
#         last = as_team1.iloc[0]
#         is_home = True
#     else:
#         if as_team1.iloc[0]["Date"] > as_team2.iloc[0]["Date"]:
#             last = as_team1.iloc[0]
#             is_home = True
#         else:
#             last = as_team2.iloc[0]
#             is_home = False
    
#     if is_home:
#         return {
#             "Elo": last["EloTeam1"],
#             "SquadValue": last["SquadValueTeam1"],
#             "RF": last["RFteamOne"],
#             "RGF": last["RGFteamOne"],
#             "RGA": last["RGAteamOne"],
#             "Days": last["DaysSinceLastTeam1"],
#             "EloDiff_contribution": 1  # positive = team1
#         }
#     else:
#         return {
#             "Elo": last["EloTeam2"],
#             "SquadValue": last["SquadValueTeam2"],
#             "RF": last["RFteamTwo"],
#             "RGF": last["RGFteamTwo"],
#             "RGA": last["RGAteamTwo"],
#             "Days": last["DaysSinceLastTeam2"],
#             "EloDiff_contribution": -1  # negative = team2
#         }

# def predict_match(team1_name, team2_name, is_neutral=True, is_knockout=False):
#     try:
#         team1_idx = TeamNames.index(team1_name)
#         team2_idx = TeamNames.index(team2_name)
#     except ValueError as e:
#         print(f"Team not found: {e}")
#         return
    
#     f1 = get_team_features(team1_name, team2_name)
#     f2 = get_team_features(team2_name, team1_name)
    
#     if f1 is None or f2 is None:
#         print("Not enough data for one of the teams")
#         return
    
#     elo_diff = f1["Elo"] - f2["Elo"]
#     squad_diff = f1["SquadValue"] - f2["SquadValue"]
    
#     # Build feature row in same order as training
#     feature_dict = {
#         "Team1": team1_idx,
#         "Team2": team2_idx,
#         "Neutral": int(is_neutral),
#         "h2hTeamOne": -1,
#         "h2hTeamTwo": -1,
#         "RFteamOne": f1["RF"],
#         "RFteamTwo": f2["RF"],
#         "RGFteamOne": f1["RGF"],
#         "RGAteamOne": f1["RGA"],
#         "RGFteamTwo": f2["RGF"],
#         "RGAteamTwo": f2["RGA"],
#         "EloTeam1": f1["Elo"],
#         "EloTeam2": f2["Elo"],
#         "SquadValueTeam1": f1["SquadValue"],
#         "SquadValueTeam2": f2["SquadValue"],
#         "DaysSinceLastTeam1": f1["Days"],
#         "DaysSinceLastTeam2": f2["Days"],
#         "EloDiff": elo_diff,
#         "SquadValueDiff": squad_diff,
#     }
    
#     # Make sure columns match training order exactly
#     model = joblib.load('group_stage_model.pkl')
#     feature_df = pd.DataFrame([feature_dict])[X.columns]
    
#     proba = model.predict_proba(feature_df)[0]
    
#     # classes are [0=draw, 1=team1, 2=team2]
#     draw_threshold = 0.36
    
#     if is_knockout:
#         # normalise between team1 and team2 only
#         team1_prob = proba[1] / (proba[1] + proba[2])
#         team2_prob = proba[2] / (proba[1] + proba[2])
#         print(f"\n{'='*40}")
#         print(f"  KNOCKOUT: {team1_name} vs {team2_name}")
#         print(f"{'='*40}")
#         print(f"  {team1_name} win:  {team1_prob*100:.1f}%")
#         print(f"  {team2_name} win:  {team2_prob*100:.1f}%")
#         if team1_prob > team2_prob:
#             print(f"\n  Predicted winner: {team1_name}")
#         else:
#             print(f"\n  Predicted winner: {team2_name}")
#     else:
#         draw_prob = proba[0]
#         team1_prob = proba[1]
#         team2_prob = proba[2]
        
#         if draw_prob > draw_threshold:
#             prediction = "Draw"
#         elif team1_prob > team2_prob:
#             prediction = f"{team1_name} win"
#         else:
#             prediction = f"{team2_name} win"
        
#         print(f"\n{'='*40}")
#         print(f"  {team1_name} vs {team2_name}")
#         print(f"{'='*40}")
#         print(f"  {team1_name} win:  {team1_prob*100:.1f}%")
#         print(f"  Draw:          {draw_prob*100:.1f}%")
#         print(f"  {team2_name} win:  {team2_prob*100:.1f}%")
#         print(f"\n  Prediction: {prediction}")
#     print(f"{'='*40}\n")

# def run_interface():
#     print("\nFootball Match Predictor")
#     print("Type 'quit' to exit\n")
    
#     while True:
#         team1 = input("Enter Team 1: ").strip()
#         if team1.lower() == 'quit':
#             break
            
#         team2 = input("Enter Team 2: ").strip()
#         if team2.lower() == 'quit':
#             break
        
#         stage = input("Group stage or knockout? (g/k): ").strip().lower()
#         is_knockout = stage == 'k'
        
#         neutral = input("Neutral venue? (y/n): ").strip().lower()
#         is_neutral = neutral == 'y'
        
#         predict_match(team1, team2, is_neutral=is_neutral, is_knockout=is_knockout)
        
#         another = input("Predict another match? (y/n): ").strip().lower()
#         if another != 'y':
#             break

# run_interface()

In [ ]:
# ============================================================
# Run this in your notebook AFTER GroupStageAI is trained.
# Produces three files for the desktop app's backend folder:
#   model.pkl           - the trained estimator
#   xcols.json          - exact feature column order
#   team_features.json  - each team's latest feature snapshot
# Copy all three into worldcup_app/backend/ next to app.py
# ============================================================

import json, joblib

# 1. model
joblib.dump(GroupStageAI.best_estimator_, "model.pkl")

# 2. exact column order the model expects
with open("xcols.json", "w") as f:
    json.dump(list(X.columns), f)

# 3. per-team latest features (reuses your get_team_features)
def latest_feature_snapshot(team):
    team_idx = TeamNames.index(team)
    as_team1 = cleanedDF[cleanedDF["Team1"] == team_idx].tail(1)
    as_team2 = cleanedDF[cleanedDF["Team2"] == team_idx].tail(1)

    if len(as_team1) == 0 and len(as_team2) == 0:
        return None
    elif len(as_team1) == 0:
        last, is_home = as_team2.iloc[0], False
    elif len(as_team2) == 0:
        last, is_home = as_team1.iloc[0], True
    else:
        if as_team1.iloc[0]["Date"] > as_team2.iloc[0]["Date"]:
            last, is_home = as_team1.iloc[0], True
        else:
            last, is_home = as_team2.iloc[0], False

    if is_home:
        elo, squad = last["EloTeam1"], last["SquadValueTeam1"]
        rf, gf, ga, days = last["RFteamOne"], last["RGFteamOne"], last["RGAteamOne"], last["DaysSinceLastTeam1"]
    else:
        elo, squad = last["EloTeam2"], last["SquadValueTeam2"]
        rf, gf, ga, days = last["RFteamTwo"], last["RGFteamTwo"], last["RGAteamTwo"], last["DaysSinceLastTeam2"]

    return {
        "idx": team_idx,
        "elo": float(elo),
        "squad": float(squad),
        "rf": float(rf),
        "gf": float(gf),
        "ga": float(ga),
        "days": int(days) if days >= 0 else -1,
    }

# all 48 teams in the tournament
WC_TEAMS = [
    "Mexico","South Africa","South Korea","Czech Republic","Canada",
    "Bosnia and Herzegovina","Qatar","Switzerland","United States","Paraguay",
    "Australia","Turkey","Brazil","Morocco","Haiti","Scotland","Germany",
    "Curaçao","Ivory Coast","Ecuador","Netherlands","Japan","Sweden","Tunisia",
    "Belgium","Egypt","Iran","New Zealand","Spain","Cape Verde","Saudi Arabia",
    "Uruguay","France","Senegal","Iraq","Norway","Argentina","Algeria",
    "Austria","Jordan","Portugal","DR Congo","Uzbekistan","Colombia","England",
    "Croatia","Ghana","Panama",
]

snap = {}
missing = []
for t in WC_TEAMS:
    s = latest_feature_snapshot(t)
    if s is None:
        missing.append(t)
    else:
        snap[t] = s

with open("team_features.json", "w") as f:
    json.dump(snap, f, indent=2)

print(f"Exported {len(snap)} teams.")
if missing:
    print("MISSING (no data):", missing)
print("Files: model.pkl, xcols.json, team_features.json")
print("Copy them into worldcup_app/backend/")


Exported 48 teams.
Files: model.pkl, xcols.json, team_features.json
Copy them into worldcup_app/backend/


: 